# 06 - Transformer with Defense 4 (Knowledge Distillation)

Teacher: baseline transformer, Student: smaller distilled model

In [1]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

from research_protocol import (
    evaluate_mia_research_protocol,
    compare_defense_vs_baseline,
    set_seed,
)
from research_protocol_robust import evaluate_mia_robust_protocol

In [ ]:
T_DISTILL = 3.0
ALPHA_HARD = 0.5
ATTACK_SEEDS = [11, 22, 33, 44, 55]

# Attaquant fort (meme base que notebook 02)
N_SHADOWS = 30
SHADOW_EPOCHS = 80
SHADOW_BATCH = 16

ROBUST_N_SHADOWS = 40
ROBUST_SHADOW_EPOCHS = 80
ROBUST_BOUNDARY_DIRS = 32
ROBUST_BOUNDARY_STEPS = 10
ROBUST_BOUNDARY_MAX_ALPHA = 2.0


def build_transformer(n_features, d_model=64, heads=4, ff=128, dense=64, dropout=0.15):
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(d_model)(inp)
    a = layers.MultiHeadAttention(num_heads=heads, key_dim=max(1, d_model // heads), dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(ff, activation='relu')(x)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(d_model)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.Flatten()(x)
    x = layers.Dense(dense, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    return m


def soften(p, t=3.0):
    p = np.clip(np.asarray(p), 1e-6, 1 - 1e-6)
    logit = np.log(p / (1 - p))
    q = 1.0 / (1.0 + np.exp(-logit / max(t, 1e-6)))
    return np.clip(q, 1e-6, 1 - 1e-6)

In [3]:
print('=== Load prepared data ===')
b = np.load('../data_preparation/prepared_oasis2_transformer.npz', allow_pickle=True)
X, y = b['X'], b['y']
X_train, X_test = b['X_train'], b['X_test']
y_train, y_test = b['y_train'], b['y_test']
X_train_seq, X_test_seq = b['X_train_seq'], b['X_test_seq']
X_shadow_raw, y_shadow = b['X_shadow_raw'], b['y_shadow']
SEED = int(b['seed'][0])
print(f'X={X.shape}, train={X_train.shape}, test={X_test.shape}, shadow={X_shadow_raw.shape}')

=== Load prepared data ===
X=(354, 9), train=(70, 9), test=(284, 9), shadow=(107, 9)


In [ ]:
print('=== Train teacher baseline (attack-target) ===')
set_seed(SEED + 10); tf.keras.backend.clear_session()
teacher = build_transformer(X_train.shape[1], d_model=64, heads=4, ff=128, dense=64, dropout=0.0)
teacher.fit(X_train_seq, y_train, epochs=260, batch_size=32, verbose=0)
teacher_acc = accuracy_score(y_test, (teacher.predict(X_test_seq, verbose=0).ravel()>=0.5).astype(int))
print(f'teacher_acc={teacher_acc:.4f}')

=== Train teacher baseline ===
teacher_acc=0.9331


In [5]:
print('=== Train student with distillation ===')
soft = soften(teacher.predict(X_train_seq, verbose=0).ravel(), T_DISTILL)
y_mix = ALPHA_HARD * y_train.astype(np.float32) + (1.0 - ALPHA_HARD) * soft.astype(np.float32)
set_seed(SEED + 20); tf.keras.backend.clear_session()
student = build_transformer(X_train.shape[1], d_model=32, heads=2, ff=64, dense=32, dropout=0.20)
es2 = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True, verbose=0)
student.fit(X_train_seq, y_mix, epochs=130, batch_size=32, validation_split=0.2, callbacks=[es2], verbose=0)
student_acc = accuracy_score(y_test, (student.predict(X_test_seq, verbose=0).ravel()>=0.5).astype(int))
print(f'student_acc={student_acc:.4f}, delta={student_acc-teacher_acc:.4f}')

=== Train student with distillation ===
student_acc=0.9155, delta=-0.0176


In [ ]:
print('=== MIA attacks on teacher (unified protocol, strong attacker) ===')
att_teacher, teacher_per_seed = evaluate_mia_research_protocol(
    target_model=teacher,
    X_train_seq=X_train_seq,
    y_train=y_train,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer(nf, d_model=64, heads=4, ff=128, dense=64, dropout=0.0),
    seed_list=ATTACK_SEEDS,
    n_shadows=N_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
)
display(att_teacher.round(4))

=== MIA attacks on teacher (unified protocol) ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
1,shadow_meta,0.5229,0.0271,0.7901,0.0059,0.0667,0.1491,0.0071,0.0160,0.0129,0.0289
2,threshold_loss,0.5120,0.0237,0.5127,0.0320,0.2137,0.0146,0.5500,0.0741,0.3073,0.0236
0,logistic,0.5082,0.0287,0.8028,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [ ]:
print('=== MIA attacks on student (Defense 4) with unified protocol ===')
att_student, student_per_seed = evaluate_mia_research_protocol(
    target_model=student,
    X_train_seq=X_train_seq,
    y_train=y_train,
    X_test_seq=X_test_seq,
    y_test=y_test,
    X_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer(nf, d_model=32, heads=2, ff=64, dense=32, dropout=0.20),
    seed_list=ATTACK_SEEDS,
    n_shadows=N_SHADOWS,
    shadow_epochs=SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
)
display(att_student.round(4))

=== MIA attacks on student (Defense 4) with unified protocol ===


,attack,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,logistic,0.5156,0.0180,0.8028,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,threshold_loss,0.5148,0.0182,0.5099,0.0519,0.1965,0.0147,0.4857,0.1201,0.2779,0.0318
1,shadow_meta,0.4984,0.0359,0.7789,0.0063,0.2071,0.0568,0.0429,0.0160,0.0707,0.0244


In [8]:
print('=== Teacher baseline vs Defense 4 ===')
cmp = compare_defense_vs_baseline(att_teacher, att_student, 'defense4')
display(cmp.round(4))

mcmp = pd.DataFrame([
    {
        'model': 'Transformer',
        'test_acc_teacher': float(teacher_acc),
        'test_acc_student': float(student_acc),
        'delta_test_acc': float(student_acc - teacher_acc),
    }
])
display(mcmp.round(4))

=== Teacher baseline vs Defense 4 ===


,attack,auc_mean_baseline,accuracy_mean_baseline,f1_mean_baseline,auc_mean_defense4,accuracy_mean_defense4,f1_mean_defense4,delta_auc,delta_accuracy,delta_f1
2,logistic,0.5082,0.8028,0.0000,0.5156,0.8028,0.0000,0.0074,0.0000,0.0000
0,shadow_meta,0.5229,0.7901,0.0129,0.4984,0.7789,0.0707,-0.0245,-0.0113,0.0577
1,threshold_loss,0.5120,0.5127,0.3073,0.5148,0.5099,0.2779,0.0028,-0.0028,-0.0293


,model,test_acc_teacher,test_acc_student,delta_test_acc
0,Transformer,0.9331,0.9155,-0.0176


In [9]:
print('=== Quick AUC summary ===')
quick_auc = cmp[['attack', 'auc_mean_baseline', 'auc_mean_defense4', 'delta_auc']].sort_values('delta_auc')
display(quick_auc.round(4))

=== Quick AUC summary ===


,attack,auc_mean_baseline,auc_mean_defense4,delta_auc
0,shadow_meta,0.5229,0.4984,-0.0245
1,threshold_loss,0.5120,0.5148,0.0028
2,logistic,0.5082,0.5156,0.0074


In [ ]:
print('=== Robust adaptive attack (meta + LiRA + boundary): teacher vs defense4 ===')

teacher_robust_summary, teacher_robust_per_seed = evaluate_mia_robust_protocol(
    target_model=teacher,
    x_train_seq=X_train_seq,
    y_train=y_train,
    x_test_seq=X_test_seq,
    y_test=y_test,
    x_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer(nf, d_model=64, heads=4, ff=128, dense=64, dropout=0.0),
    seed_list=ATTACK_SEEDS,
    n_shadows=ROBUST_N_SHADOWS,
    shadow_epochs=ROBUST_SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
    boundary_dirs=ROBUST_BOUNDARY_DIRS,
    boundary_steps=ROBUST_BOUNDARY_STEPS,
    boundary_max_alpha=ROBUST_BOUNDARY_MAX_ALPHA,
)

student_robust_summary, student_robust_per_seed = evaluate_mia_robust_protocol(
    target_model=student,
    x_train_seq=X_train_seq,
    y_train=y_train,
    x_test_seq=X_test_seq,
    y_test=y_test,
    x_shadow_raw=X_shadow_raw,
    y_shadow=y_shadow,
    model_builder=lambda nf: build_transformer(nf, d_model=32, heads=2, ff=64, dense=32, dropout=0.20),
    seed_list=ATTACK_SEEDS,
    n_shadows=ROBUST_N_SHADOWS,
    shadow_epochs=ROBUST_SHADOW_EPOCHS,
    shadow_batch_size=SHADOW_BATCH,
    boundary_dirs=ROBUST_BOUNDARY_DIRS,
    boundary_steps=ROBUST_BOUNDARY_STEPS,
    boundary_max_alpha=ROBUST_BOUNDARY_MAX_ALPHA,
)

cmp_robust = compare_defense_vs_baseline(teacher_robust_summary, student_robust_summary, 'defense4')
print('=== Quick AUC summary (robust adaptive) ===')
quick_auc_robust = cmp_robust[['attack', 'auc_mean_baseline', 'auc_mean_defense4', 'delta_auc']].sort_values('delta_auc')
display(quick_auc_robust.round(4))

=== Robust adaptive attack (meta + LiRA + boundary): teacher vs defense4 ===
=== Quick AUC summary (robust adaptive) ===


,attack,auc_mean_baseline,auc_mean_defense4,delta_auc
0,shadow_meta,0.5138,0.5146,0.0008
